# 05 - High-Risk Complaint Triage
**Author:** Sanjeev Kumar | IIT Bombay

Predict priority-review complaints using untimely response as proxy.

**Key:** Never use accuracy! Use PR-AUC, Recall@K, Lift@K!

In [ ]:
import matplotlib
matplotlib.use('Agg')
import pandas as pd, numpy as np, re, os, sys, warnings, joblib
import matplotlib.pyplot as plt
warnings.filterwarnings('ignore')
os.makedirs('../reports', exist_ok=True)
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import average_precision_score, roc_auc_score, brier_score_loss, precision_recall_curve
from xgboost import XGBClassifier
print("Imports done!")

In [ ]:
PRODUCT_MAP = {
    'Credit reporting or other personal consumer reports': 'Credit Reporting',
    'Credit reporting, credit repair services, or other personal consumer reports': 'Credit Reporting',
    'Credit reporting': 'Credit Reporting', 'Debt collection': 'Debt Collection',
    'Mortgage': 'Mortgage', 'Checking or savings account': 'Bank Account',
    'Bank account or service': 'Bank Account', 'Credit card': 'Credit Card',
    'Credit card or prepaid card': 'Credit Card', 'Prepaid card': 'Credit Card',
    'Student loan': 'Loans', 'Vehicle loan or lease': 'Loans', 'Consumer Loan': 'Loans',
    'Payday loan, title loan, personal loan, or advance loan': 'Loans',
    'Payday loan, title loan, or personal loan': 'Loans', 'Payday loan': 'Loans',
    'Money transfer, virtual currency, or money service': 'Money Transfer',
    'Money transfers': 'Money Transfer', 'Debt or credit management': 'Debt Collection',
    'Other financial service': 'Other',
}
def clean_text(t):
    if pd.isna(t): return ""
    t = str(t).lower(); t = re.sub(r'x{2,}', 'XXXX', t)
    return re.sub(r'\s+', ' ', t).strip()

os.makedirs('../data', exist_ok=True)
local_path = '../data/complaints_200k.csv'
if os.path.exists(local_path):
    print("Loading from local cache...")
    df = pd.read_csv(local_path, low_memory=False)
else:
    import requests
    url = "https://files.consumerfinance.gov/ccdb/complaints.csv.zip"
    zip_path = '../data/complaints.csv.zip'
    print("Downloading CFPB data (streaming)...")
    with requests.get(url, stream=True, timeout=600) as r:
        r.raise_for_status()
        with open(zip_path, 'wb') as f:
            for chunk in r.iter_content(chunk_size=1024 * 1024):
                if chunk:
                    f.write(chunk)
    df = pd.read_csv(zip_path, compression='zip', low_memory=False, nrows=200000)
    df.to_csv(local_path, index=False)

df_text = df[df['Consumer complaint narrative'].notna()].copy()
df_text['product_clean'] = df_text['Product'].map(PRODUCT_MAP).fillna('Other')
df_text['narrative_clean'] = df_text['Consumer complaint narrative'].apply(clean_text)
df_text['Date received'] = pd.to_datetime(df_text['Date received'])
df_text['is_priority_review'] = (df_text['Timely response?'] == 'No').astype(int)
df_sorted = df_text.sort_values('Date received').reset_index(drop=True)
n = len(df_sorted)
train_df = df_sorted.iloc[:int(n * 0.70)]
val_df   = df_sorted.iloc[int(n * 0.70):int(n * 0.85)]
test_df  = df_sorted.iloc[int(n * 0.85):]
X_train = train_df['narrative_clean'].values
X_val   = val_df['narrative_clean'].values
X_test  = test_df['narrative_clean'].values
y_train_risk = train_df['is_priority_review'].values
y_val_risk   = val_df['is_priority_review'].values
y_test_risk  = test_df['is_priority_review'].values
print(f"Train positives: {y_train_risk.sum()} ({y_train_risk.mean():.2%})")
print(f"Val positives:   {y_val_risk.sum()} ({y_val_risk.mean():.2%})")
print(f"Test positives:  {y_test_risk.sum()} ({y_test_risk.mean():.2%})")

In [ ]:
def triage_metrics(y_true, y_prob, name):
    y_true = np.array(y_true); y_prob = np.array(y_prob)
    pr_auc  = average_precision_score(y_true, y_prob)
    roc_auc = roc_auc_score(y_true, y_prob)
    brier   = brier_score_loss(y_true, y_prob)
    print(f"\n{name}:")
    print(f"  PR-AUC:  {pr_auc:.4f} (random={y_true.mean():.4f}, {pr_auc/y_true.mean():.1f}x lift)")
    print(f"  ROC-AUC: {roc_auc:.4f}")
    print(f"  Brier:   {brier:.4f}")
    n = len(y_true)
    sorted_idx = np.argsort(y_prob)[::-1]
    for pct in [0.05, 0.10, 0.20]:
        k = int(n * pct)
        top_k = y_true[sorted_idx[:k]]
        recall_k = top_k.sum() / y_true.sum()
        prec_k   = top_k.mean()
        lift_k   = recall_k / pct
        print(f"  Top {pct:.0%}: Recall={recall_k:.1%}  Precision={prec_k:.1%}  Lift={lift_k:.1f}x")
    return pr_auc, roc_auc

In [ ]:
# TRIAGE MODEL 1: TF-IDF + LR
# Re-use the TF-IDF fitted in notebook 02 — transform only, no re-fit.
# Re-fitting 50k bigrams exceeds available RAM on this machine.
print("=" * 60)
print("TRIAGE MODEL 1: TF-IDF + LR (class_weight=balanced)")
fitted_tfidf = joblib.load('../models/tfidf_vectorizer.pkl')
X_tr_tf = fitted_tfidf.transform(X_train)
X_va_tf = fitted_tfidf.transform(X_val)
X_te_tf = fitted_tfidf.transform(X_test)
print(f"Feature matrix: {X_tr_tf.shape}")
triage_lr = LogisticRegression(max_iter=1000, C=1.0, class_weight='balanced', random_state=42)
triage_lr.fit(X_tr_tf, y_train_risk)
lr_val_prauc,  _ = triage_metrics(y_val_risk,  triage_lr.predict_proba(X_va_tf)[:,1],  "LR Val")
lr_test_prauc, _ = triage_metrics(y_test_risk, triage_lr.predict_proba(X_te_tf)[:,1], "LR Test")

In [ ]:
# TRIAGE MODEL 2: XGBoost
# Uses a compact 5k-unigram TF-IDF and tree_method='hist' to stay within RAM.
# (XGBoost QuantileDMatrix OOMs at 50k features on this machine.)
print("\n" + "=" * 60)
print("TRIAGE MODEL 2: XGBoost (tuning scale_pos_weight on val)")
tfidf_xgb = TfidfVectorizer(max_features=5000, ngram_range=(1, 1), min_df=5, sublinear_tf=True)
X_tr_xgb = tfidf_xgb.fit_transform(X_train)
X_va_xgb = tfidf_xgb.transform(X_val)
X_te_xgb = tfidf_xgb.transform(X_test)
print(f"XGBoost feature matrix: {X_tr_xgb.shape}")

best_spw, best_pr, best_xgb = 30, 0, None
for spw in [30, 50, 70, 90]:
    xgb = XGBClassifier(n_estimators=100, max_depth=4, learning_rate=0.1,
                         scale_pos_weight=spw, tree_method='hist',
                         random_state=42, verbosity=0)
    xgb.fit(X_tr_xgb, y_train_risk)
    pr = average_precision_score(y_val_risk, xgb.predict_proba(X_va_xgb)[:, 1])
    print(f"  scale_pos_weight={spw:>3}: Val PR-AUC={pr:.4f}")
    if pr > best_pr:
        best_pr = pr; best_spw = spw; best_xgb = xgb

xgb_test_prauc, _ = triage_metrics(y_test_risk,
                                    best_xgb.predict_proba(X_te_xgb)[:, 1],
                                    f"XGBoost spw={best_spw} Test")

In [ ]:
fig, ax = plt.subplots(figsize=(10, 6))
for name, y_prob in [
    ('TF-IDF + LR',             triage_lr.predict_proba(X_te_tf)[:, 1]),
    (f'XGBoost spw={best_spw}', best_xgb.predict_proba(X_te_xgb)[:, 1]),
]:
    p, r, _ = precision_recall_curve(y_test_risk, y_prob)
    pr_auc  = average_precision_score(y_test_risk, y_prob)
    ax.plot(r, p, label=f"{name} (PR-AUC={pr_auc:.4f})", linewidth=2)
ax.axhline(y=y_test_risk.mean(), color='gray', linestyle='--',
           label=f"Random ({y_test_risk.mean():.3f})")
ax.set_xlabel('Recall'); ax.set_ylabel('Precision')
ax.set_title('PR Curve - High-Risk Complaint Triage', fontweight='bold')
ax.legend(); ax.grid(alpha=0.3)
plt.tight_layout()
plt.savefig('../reports/05_pr_curve.png', dpi=150, bbox_inches='tight')
plt.close()

In [ ]:
import json
os.makedirs('../models', exist_ok=True)
joblib.dump(triage_lr,    '../models/triage_lr.pkl')
joblib.dump(best_xgb,     '../models/triage_xgb.pkl')
joblib.dump(fitted_tfidf, '../models/triage_tfidf.pkl')
joblib.dump(tfidf_xgb,    '../models/triage_tfidf_xgb.pkl')

results = {
    'triage_lr_prauc':  float(lr_test_prauc),
    'triage_xgb_prauc': float(xgb_test_prauc),
    'positive_rate':    float(y_test_risk.mean()),
    'best_spw':         int(best_spw),
}
with open('../reports/triage_results.json', 'w') as f:
    json.dump(results, f, indent=2)

print("=" * 60)
print("TRIAGE COMPARISON")
print(f"Random baseline PR-AUC:  {y_test_risk.mean():.4f}")
print(f"TF-IDF + LR PR-AUC:      {lr_test_prauc:.4f}")
print(f"XGBoost PR-AUC:          {xgb_test_prauc:.4f}")
print("\nModels saved!")

## Business Impact Statement

> 'By reviewing the top 10% highest-risk complaints, our model captures significantly more priority-review cases than random review — giving meaningful lift for operations teams.'

### Leakage Controls
- **Allowed at prediction time:** complaint narrative text, submission channel, state
- **NOT allowed:** company response, timely response flag, any post-resolution fields

**Next → 06_final_evaluation.ipynb**